# Transfer Learning: Classificacao de Gatos e Cachorros

Este notebook faz parte de um projeto para o desafio da DIO sobre Transfer Learning com Deep Learning.

A ideia e construir um classificador de imagens capaz de diferenciar gatos e cachorros utilizando Python, TensorFlow, Keras e a arquitetura MobileNetV2 com pesos pre-treinados na ImageNet.

## Visao geral

Neste projeto vamos:

- carregar o dataset `cats_vs_dogs` com `tensorflow_datasets`;
- dividir os dados em treino, validacao e teste;
- redimensionar e normalizar as imagens;
- aplicar data augmentation;
- usar a MobileNetV2 como modelo base;
- treinar uma cabeca de classificacao binaria;
- avaliar o modelo e visualizar os resultados.

## 1. Importacoes

As importacoes ficam concentradas no inicio para facilitar manutencao e reproducibilidade do notebook.

In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow:', tf.__version__)
print('TensorFlow Datasets:', tfds.__version__)

## 2. Configuracoes iniciais

Definimos algumas constantes do projeto. O tamanho `160x160` e uma escolha comum para MobileNetV2 em exemplos didaticos, equilibrando qualidade visual e custo computacional.

In [ ]:
SEED = 42
IMG_SIZE = (160, 160)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Carregamento do dataset

O `tensorflow_datasets` permite baixar e preparar o dataset de forma padronizada. Aqui usamos uma divisao explicita: 80% para treino, 10% para validacao e 10% para teste.

In [ ]:
(train_ds, val_ds, test_ds), metadata = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'],
    with_info=True,
    as_supervised=True,
    shuffle_files=True,
)

class_names = metadata.features['label'].names
num_train = tf.data.experimental.cardinality(train_ds).numpy()
num_val = tf.data.experimental.cardinality(val_ds).numpy()
num_test = tf.data.experimental.cardinality(test_ds).numpy()

print('Classes:', class_names)
print('Imagens de treino:', num_train)
print('Imagens de validacao:', num_val)
print('Imagens de teste:', num_test)

## 4. Visualizacao de amostras

Antes de treinar qualquer modelo, e importante olhar os dados. Essa etapa ajuda a entender variacoes de tamanho, iluminacao, enquadramento e possiveis ruidos do dataset.

In [ ]:
plt.figure(figsize=(10, 10))

for index, (image, label) in enumerate(train_ds.take(9)):
    ax = plt.subplot(3, 3, index + 1)
    plt.imshow(image)
    plt.title(class_names[label.numpy()])
    plt.axis('off')

plt.tight_layout()
plt.show()

## 5. Pre-processamento

A MobileNetV2 espera imagens em um formato consistente. Vamos redimensionar todas as imagens para `160x160` e converter os labels para `float32`, formato adequado para classificacao binaria com `binary_crossentropy`.

In [ ]:
def preprocess_image(image, label):
    """Redimensiona a imagem e prepara o label para classificacao binaria."""
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    label = tf.cast(label, tf.float32)
    return image, label


train_batches = (
    train_ds
    .shuffle(1000, seed=SEED)
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_batches = (
    val_ds
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

test_batches = (
    test_ds
    .map(preprocess_image, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

## 6. Data augmentation

Data augmentation cria pequenas variacoes artificiais nas imagens de treino. Isso ajuda o modelo a generalizar melhor, reduzindo a chance de memorizar caracteristicas especificas do conjunto de treinamento.

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ],
    name='data_augmentation',
)

Abaixo visualizamos como uma mesma imagem pode ser transformada pela camada de aumento de dados durante o treinamento.

In [ ]:
for image_batch, label_batch in train_batches.take(1):
    first_image = image_batch[0]
    break

plt.figure(figsize=(10, 10))

for index in range(9):
    augmented_image = data_augmentation(tf.expand_dims(first_image, axis=0), training=True)
    ax = plt.subplot(3, 3, index + 1)
    plt.imshow(tf.cast(augmented_image[0], tf.uint8))
    plt.axis('off')

plt.tight_layout()
plt.show()

## 7. Normalizacao para MobileNetV2

A MobileNetV2 pre-treinada no Keras trabalha melhor quando as imagens seguem o mesmo pre-processamento usado no treinamento original. A funcao `preprocess_input` transforma os pixels para a escala esperada pela arquitetura.

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

## 8. Modelo base: MobileNetV2

Carregamos a MobileNetV2 sem a camada final de classificacao (`include_top=False`). Assim, usamos apenas a parte convolucional como extratora de caracteristicas.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet',
)

# Congelamos o modelo base para preservar os pesos aprendidos na ImageNet.
base_model.trainable = False

base_model.summary()

## 9. Criacao da cabeca de classificacao

A cabeca de classificacao recebe as caracteristicas extraidas pela MobileNetV2 e produz uma saida binaria. Como usamos `BinaryCrossentropy(from_logits=True)`, a ultima camada retorna logits, sem funcao sigmoid aplicada diretamente.

In [ ]:
inputs = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, name='classifier')(x)

model = keras.Model(inputs, outputs, name='cats_vs_dogs_mobilenetv2')

model.summary()

## 10. Compilacao

Usamos Adam com uma taxa de aprendizado conservadora. Para classificacao binaria, `BinaryCrossentropy` e uma escolha direta e adequada.

In [ ]:
initial_learning_rate = 0.0001

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=initial_learning_rate),
    loss=keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

## 11. Treinamento

Treinamos apenas a cabeca de classificacao. O modelo base permanece congelado, o que torna o treinamento mais rapido e reduz o risco de destruir os pesos pre-treinados.

In [ ]:
EPOCHS = 10

history = model.fit(
    train_batches,
    epochs=EPOCHS,
    validation_data=val_batches,
)

## 12. Avaliacao final

A avaliacao no conjunto de teste fornece uma estimativa mais realista de desempenho, pois o modelo nao usou essas imagens durante o treinamento nem durante a validacao.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_batches)

print(f'Loss no teste: {test_loss:.4f}')
print(f'Acuracia no teste: {test_accuracy:.4f}')

## 13. Graficos de acuracia e loss

Os graficos ajudam a analisar o comportamento do treinamento. Uma grande diferenca entre treino e validacao pode indicar overfitting.

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(EPOCHS)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Treino')
plt.plot(epochs_range, val_acc, label='Validacao')
plt.title('Acuracia')
plt.xlabel('Epoca')
plt.ylabel('Acuracia')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Treino')
plt.plot(epochs_range, val_loss, label='Validacao')
plt.title('Loss')
plt.xlabel('Epoca')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

## 14. Predicoes em imagens de exemplo

Para interpretar a saida do modelo, aplicamos sigmoid aos logits. Valores proximos de 0 indicam gato, e valores proximos de 1 indicam cachorro, seguindo a codificacao do dataset.

In [ ]:
plt.figure(figsize=(12, 12))

for image_batch, label_batch in test_batches.take(1):
    logits = model.predict(image_batch)
    probabilities = tf.sigmoid(logits).numpy().flatten()

    for index in range(9):
        probability = probabilities[index]
        predicted_label = 1 if probability >= 0.5 else 0
        true_label = int(label_batch[index].numpy())
        color = 'green' if predicted_label == true_label else 'red'

        ax = plt.subplot(3, 3, index + 1)
        plt.imshow(tf.cast(image_batch[index], tf.uint8))
        plt.title(
            f'Real: {class_names[true_label]}\nPrevisto: {class_names[predicted_label]} ({probability:.2f})',
            color=color,
        )
        plt.axis('off')

plt.tight_layout()
plt.show()

## 15. Opcional: fine-tuning

Depois que a cabeca de classificacao aprende uma boa separacao inicial, uma melhoria comum e descongelar as ultimas camadas do modelo base e continuar o treinamento com uma taxa de aprendizado menor.

Esta etapa e opcional porque aumenta o tempo de treino e exige mais cuidado para evitar overfitting.

In [ ]:
# Exemplo de fine-tuning opcional.
# Altere para True apenas se quiser experimentar uma segunda etapa de treinamento.

RUN_FINE_TUNING = False

if RUN_FINE_TUNING:
    base_model.trainable = True

    # Mantemos as primeiras camadas congeladas e ajustamos apenas as camadas finais.
    fine_tune_at = 100

    for layer in base_model.layers[:fine_tune_at]:
        layer.trainable = False

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=initial_learning_rate / 10),
        loss=keras.losses.BinaryCrossentropy(from_logits=True),
        metrics=['accuracy'],
    )

    fine_tune_epochs = 5
    total_epochs = EPOCHS + fine_tune_epochs

    history_fine = model.fit(
        train_batches,
        epochs=total_epochs,
        initial_epoch=history.epoch[-1] + 1,
        validation_data=val_batches,
    )

## Conclusao

Com poucas etapas, conseguimos construir um classificador de imagens usando Transfer Learning. A MobileNetV2 fornece uma base visual forte, enquanto a cabeca de classificacao adapta o modelo ao problema especifico de gatos e cachorros.

Esse fluxo e uma boa base para outros projetos de classificacao de imagens: basta trocar o dataset, ajustar o numero de classes e revisar a camada final do modelo.